In [1]:
import os
from utils.dataloader import ROOT
os.chdir(ROOT)  # write data/artifacts/ at the repo root

from pathlib import Path
from utils.dataloader import load_results
from utils import detection, tables

df = detection.add_n15_label(load_results())
print(f"{len(df)} rows · models: {sorted(df['model_alias'].unique())}")

11554 rows · models: ['deepseek-v4-flash', 'gemini35-flash', 'gpt54-nano', 'haiku45']


# Detection effectiveness

How well does each model **detect** the target smell? We compare the model's
prediction (`smell_detected`) against ground truth, reporting precision, recall
and F1 (positive class = smell present).

**Scope.** rep1 (first execution) only. The tables below use the balanced
**n15** ground truth (`gt_n15`, 570 pairs = 285 present / 285 absent): every
model is scored on the *same* pairs, so F1 is comparable across models (paper
`tab:rq1_overall_detection` headline). The per-instance `ground_truth` label instead scores each
model on its own corpus — DeepSeek ran the full 1710-pair GT, the rest the n15
subset — so pass `label="ground_truth"` to inspect it, but those F1s are **not**
cross-model-comparable.

In [2]:
# F1 per model — apples-to-apples headline (n15 common scope, rep1)
f1_by_model = detection.metrics_by(df, "model_alias")
f1_by_model

,model_alias,n,precision,recall,f1,accuracy,TP,FP,FN,TN
0,deepseek-v4-flash,1140,0.935039,0.833333,0.881262,0.887719,475,33,95,537
1,gemini35-flash,1140,0.961538,0.789474,0.867052,0.878947,450,18,120,552
2,gpt54-nano,1140,0.943601,0.763158,0.843841,0.858772,435,26,135,544
3,haiku45,1140,0.933981,0.843860,0.886636,0.892105,481,34,89,536


In [3]:
# F1 per model x design
f1_by_design = detection.metrics_by(df, "model_alias", "design")
f1_by_design

,model_alias,design,n,precision,recall,f1,accuracy,TP,FP,FN,TN
0,deepseek-v4-flash,D1,570,0.940476,0.831579,0.882682,0.889474,237,15,48,270
1,deepseek-v4-flash,D2,570,0.929688,0.835088,0.879852,0.885965,238,18,47,267
2,gemini35-flash,D1,570,0.982301,0.778947,0.868885,0.882456,222,4,63,281
3,gemini35-flash,D2,570,0.942149,0.800000,0.865275,0.875439,228,14,57,271
4,gpt54-nano,D1,570,0.961722,0.705263,0.813765,0.838596,201,8,84,277
5,gpt54-nano,D2,570,0.928571,0.821053,0.871508,0.878947,234,18,51,267
6,haiku45,D1,570,0.930769,0.849123,0.888073,0.892982,242,18,43,267
7,haiku45,D2,570,0.937255,0.838596,0.885185,0.891228,239,16,46,269


In [4]:
# F1 per model x smell
f1_by_smell = detection.metrics_by(df, "model_alias", "target_smell")
f1_by_smell

,model_alias,target_smell,n,precision,recall,f1,accuracy,TP,FP,FN,TN
0,deepseek-v4-flash,MissingConcurrencyPR,60,1.000000,1.000000,1.000000,1.000000,30,0,0,30
1,deepseek-v4-flash,MissingJobTimeout,60,0.612245,1.000000,0.759494,0.683333,30,19,0,11
2,deepseek-v4-flash,UnsafeDeployOnFork,60,1.000000,0.733333,0.846154,0.866667,22,0,8,30
3,deepseek-v4-flash,MissingForkGuardSchedule,60,1.000000,1.000000,1.000000,1.000000,30,0,0,30
4,deepseek-v4-flash,MissingTokenPermissions,60,1.000000,1.000000,1.000000,1.000000,30,0,0,30
...,...,...,...,...,...,...,...,...,...,...,...
71,haiku45,MissingForkGuardPR,60,1.000000,0.400000,0.571429,0.700000,12,0,18,30
72,haiku45,MutableRunnerVersion,60,1.000000,0.800000,0.888889,0.900000,24,0,6,30
73,haiku45,SingleOSExecution,60,0.866667,0.433333,0.577778,0.683333,13,2,17,28
74,haiku45,UnpinnedActionVersion,60,1.000000,0.666667,0.800000,0.833333,20,0,10,30


In [5]:
_, output_path = tables.overall_detection_table(df)
print(output_path)

data/artifacts/rq1_overall_detection.tex


In [6]:
smell_output_path = Path('data/artifacts/rq1_smell_level_detection_effectiveness.tex')
tables.smell_level_detection_effectiveness_table(df, smell_output_path)
print(smell_output_path)

data/artifacts/rq1_smell_level_detection_effectiveness.tex


## Stability across repetitions

GPT-5.4-nano and Claude Haiku 4.5 ran three times (k = 3). The **accuracy spread** of a
(model, design) pair is its maximum minus its minimum detection accuracy over the three
runs; the paper reports the largest spread. Models that ran once are left out.

In [7]:
spread = detection.accuracy_spread(df)
spread.to_csv("data/artifacts/rq1_accuracy_spread.csv", index=False)
print(f"maximum accuracy spread: {spread['spread'].max():.2%}")
spread

maximum accuracy spread: 2.11%


,model_alias,design,rep1,rep2,rep3,spread
0,gpt54-nano,D1,0.838596,0.826316,0.847368,0.021053
1,gpt54-nano,D2,0.878947,0.873684,0.877193,0.005263
2,haiku45,D1,0.892982,0.898246,0.894737,0.005263
3,haiku45,D2,0.891228,0.891228,0.894737,0.003509
